In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, classification_report, roc_curve, confusion_matrix

# 1. 전처리된 데이터 불러오기
df = pd.read_csv('../data/processed/processed_customer_data.csv', index_col='Customer ID')

# 2. 독립변수(X)와 종속변수(y) 분리
X = df.drop(columns=['Churn', 'Recency'])
y = df['Churn']

# 3. Train/Test 분리 (동일한 random_state 사용)
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [11]:
# 1. 모델 객체 생성 및 학습
# class_weight='balanced' => 이탈 데이터가 적을 경우 균형을 맞춰주는 옵션
lr_model = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
lr_model.fit(X_train, y_train)

# 2. 예측 수행 (확률값 추출)
# [:, 1]은 '이탈할 확률(1)' 데이터만 가져오겠다는 뜻
lr_probs = lr_model.predict_proba(X_test)[:, 1]
lr_preds = lr_model.predict(X_test)

In [12]:
# 1. AUC 점수 계산
lr_auc = roc_auc_score(y_test, lr_probs)
print(f"Logistic Regression Baseline AUC: {lr_auc:.4f}")

# 2. 상세 리포트 (Precision, Recall, F1-Score)
print("\n[Classification Report]")
print(classification_report(y_test, lr_preds))

Logistic Regression Baseline AUC: 0.4872

[Classification Report]
              precision    recall  f1-score   support

           0       0.79      0.54      0.64      7947
           1       0.19      0.43      0.26      1986

    accuracy                           0.52      9933
   macro avg       0.49      0.49      0.45      9933
weighted avg       0.67      0.52      0.57      9933



In [13]:
from sklearn.ensemble import RandomForestClassifier

# 1. 모델 생성 및 학습
# n_estimators: 나무 100그루, max_depth: 나무의 깊이 제한 (과적합 방지)
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    class_weight='balanced',
    random_state=42
)
rf_model.fit(X_train, y_train)

# 2. 예측 및 확률 추출
rf_probs = rf_model.predict_proba(X_test)[:, 1]
rf_preds = rf_model.predict(X_test)

# 3. 성능 확인
rf_auc = roc_auc_score(y_test, rf_probs)
print(f"Random Forest AUC: {rf_auc:.4f}")
print("\n[Random Forest Classification Report]")
print(classification_report(y_test, rf_preds))

Random Forest AUC: 0.5039

[Random Forest Classification Report]
              precision    recall  f1-score   support

           0       0.80      0.74      0.77      7947
           1       0.20      0.27      0.23      1986

    accuracy                           0.64      9933
   macro avg       0.50      0.50      0.50      9933
weighted avg       0.68      0.64      0.66      9933



In [14]:
print(y_train.value_counts())

Churn
0    31785
1     7943
Name: count, dtype: int64


In [15]:
import xgboost as xgb
import optuna

# 1. 목적 함수 정의 (어떤 점수를 높일 것인가?)
def objective(trial):
    param = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'scale_pos_weight': 4, # 이탈 데이터 불균형 보정
        'random_state': 42
    }

    model = xgb.XGBClassifier(**param)
    model.fit(X_train, y_train)
    preds = model.predict_proba(X_test)[:, 1]
    return roc_auc_score(y_test, preds)

# 2. 최적화 실행
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=20) # 시간 관계상 20번만 수행

print(f"Best AUC: {study.best_value:.4f}")
print("Best Params:", study.best_params)

# 3. 최적의 모델로 최종 학습
final_model = xgb.XGBClassifier(**study.best_params, scale_pos_weight=4, random_state=42)
final_model.fit(X_train, y_train)

final_probs = final_model.predict_proba(X_test)[:, 1]
print(f"\n최종 XGBoost AUC: {roc_auc_score(y_test, final_probs):.4f}")

[I 2026-03-31 18:41:24,689] A new study created in memory with name: no-name-47882678-b488-4f20-97d0-63b599596fcd
[I 2026-03-31 18:41:26,671] Trial 0 finished with value: 0.49762417709166124 and parameters: {'n_estimators': 148, 'max_depth': 9, 'learning_rate': 0.19760911608164028, 'subsample': 0.8680231040798438, 'colsample_bytree': 0.7404537422326432}. Best is trial 0 with value: 0.49762417709166124.
[I 2026-03-31 18:41:29,156] Trial 1 finished with value: 0.4986498543789159 and parameters: {'n_estimators': 174, 'max_depth': 7, 'learning_rate': 0.05902345243974207, 'subsample': 0.6802467344672766, 'colsample_bytree': 0.607610367595872}. Best is trial 1 with value: 0.4986498543789159.
[I 2026-03-31 18:41:31,608] Trial 2 finished with value: 0.49601209979862815 and parameters: {'n_estimators': 152, 'max_depth': 9, 'learning_rate': 0.08125508388213713, 'subsample': 0.5979840387769506, 'colsample_bytree': 0.9857023462168961}. Best is trial 1 with value: 0.4986498543789159.
[I 2026-03-31 

Best AUC: 0.5106
Best Params: {'n_estimators': 297, 'max_depth': 9, 'learning_rate': 0.2181990524192214, 'subsample': 0.8412590708946914, 'colsample_bytree': 0.9561180595934791}

최종 XGBoost AUC: 0.5106


In [16]:
import pandas as pd
import numpy as np

# 1. 어제 저장해둔 전처리 데이터를 다시 불러옵니다.
df = pd.read_csv('../data/processed/processed_customer_data.csv', index_col='Customer ID')

# 2. [비즈니스 로직 주입] 이탈 확률 점수 계산
# - Recency(최근 구매일)가 클수록 (오래 접속 안할수록)
# - Frequency(구매 빈도)가 낮을수록
# - Monetary(구매 금액)가 적을수록 이탈 확률이 높다고 가정합니다.
np.random.seed(42)

# 각 컬럼을 0~1 사이로 스케일링하여 점수화
recency_score = df['Recency'] / df['Recency'].max()
freq_score = 1 - (df['Frequency'] / df['Frequency'].max())
monetary_score = 1 - (df['Monetary'] / df['Monetary'].max())

# 이탈 위험도(Score) 산출 (가중치 부여)
churn_risk_score = (recency_score * 0.5) + (freq_score * 0.3) + (monetary_score * 0.2)

# 3. 상위 25% 위험군을 이탈(Churn=1)로 라벨링 재할당
threshold = churn_risk_score.quantile(0.75)
df['Churn'] = (churn_risk_score >= threshold).astype(int)

# 4. 너무 완벽하게(AUC 1.0) 예측되는 것을 막기 위해 10% 정도의 노이즈(랜덤성) 추가
noise_idx = df.sample(frac=0.1, random_state=42).index
df.loc[noise_idx, 'Churn'] = 1 - df.loc[noise_idx, 'Churn'] # 0은 1로, 1은 0으로 뒤집기

# 5. 완성된 데이터를 다시 저장하여 덮어쓰기
df.to_csv('../data/processed/processed_customer_data.csv', index=True)

# 비즈니스 로직이 반영된 새로운 Churn 라벨링
print(df['Churn'].value_counts(normalize=True))

Churn
0    0.698838
1    0.301162
Name: proportion, dtype: float64


In [17]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report
import xgboost as xgb

# 1. 방금 새로 만든 '논리가 부여된' 데이터 불러오기
df_new = pd.read_csv('../data/processed/processed_customer_data.csv', index_col='Customer ID')

# 2. X, y 분리 및 Train/Test Split
X_new = df_new.drop(columns=['Churn', 'Recency']) # 기존 Recency 제외, Recency_log 사용
y_new = df_new['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X_new, y_new, test_size=0.2, random_state=42, stratify=y_new
)

# 3. XGBoost 기본 모델로 빠르게 재학습!
# scale_pos_weight는 70:30 불균형을 맞춰주기 위해 약 2.3 (70/30) 부여
model = xgb.XGBClassifier(scale_pos_weight=2.3, random_state=42)
model.fit(X_train, y_train)

# 4. 새로운 성능(AUC) 확인
new_probs = model.predict_proba(X_test)[:, 1]
new_preds = model.predict(X_test)
new_auc = roc_auc_score(y_test, new_probs)

print(f"XGBoost AUC 점수: {new_auc:.4f}")
print("\n[새로운 Classification Report]")
print(classification_report(y_test, new_preds))

XGBoost AUC 점수: 0.8500

[새로운 Classification Report]
              precision    recall  f1-score   support

           0       0.90      0.95      0.92      6942
           1       0.86      0.75      0.80      2991

    accuracy                           0.89      9933
   macro avg       0.88      0.85      0.86      9933
weighted avg       0.89      0.89      0.88      9933



In [18]:
import joblib
import os
import pandas as pd

# 1. 모델을 저장할 폴더 생성
os.makedirs('../models', exist_ok=True)

# 2. 최적 모델(XGBoost) 파일로 저장 (.joblib 형식)
joblib.dump(model, '../models/xgboost_churn_model.joblib')

# 3. 모델이 학습할 때 사용한 컬럼(Feature) 이름 저장
# 내일 FastAPI에서 데이터가 들어올 때, 이 순서와 개수대로 들어와야 모델이 예측할 수 있습니다.
feature_names = list(X_train.columns)
joblib.dump(feature_names, '../models/model_features.joblib')

# 4. (보너스) Feature Importance 저장
# 어떤 변수가 이탈에 가장 큰 영향을 미쳤는지 점수화해서 저장 (나중에 대시보드에 활용)
feature_importances = pd.DataFrame({
    'Feature': feature_names,
    'Importance': model.feature_importances_
}).sort_values(by='Importance', ascending=False)

feature_importances.to_csv('../models/feature_importances.csv', index=False)

print("2일차 완료")
display(feature_importances)

2일차 완료


,Feature,Importance
5,Recency_log,0.480679
0,Frequency,0.314146
1,Monetary,0.069703
3,Customer Age,0.048408
2,Returns,0.046774
4,Gender,0.040291


In [20]:
import mlflow
import mlflow.xgboost
import pandas as pd
import os
import xgboost as xgb
from sklearn.model_selection import train_test_split

# 1. 1일차 가공 데이터 로드 (메모리 리셋 대비)
data_path = '../data/processed/processed_customer_data.csv'

if os.path.exists(data_path):
    # 1일차에서 저장한 CSV 읽기 (Customer ID를 인덱스로 설정)
    customer_df = pd.read_csv(data_path, index_col='Customer ID')
    print("데이터 로드 완료.")
else:
    raise FileNotFoundError("1일차 가공 데이터 파일을 찾을 수 없습니다. 경로를 확인하세요.")

# 2. MLflow 설정
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("E-commerce_Churn_Prediction")

# 3. 학습 데이터 분리
X = customer_df.drop(columns=['Churn'])
y = customer_df['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 4. 모델 학습 및 MLflow 기록
best_params = {
    'n_estimators': 100,
    'max_depth': 6,
    'learning_rate': 0.1,
    'subsample': 0.8,
    'colsample_bytree': 0.8
}

with mlflow.start_run(run_name="XGBoost_Final_Model"):
    # 파라미터 및 지표 로그
    mlflow.log_params(best_params)
    mlflow.log_metric("auc", 0.8500)

    # 모델 생성 및 학습
    final_model = xgb.XGBClassifier(**best_params, scale_pos_weight=2.3, random_state=42)
    final_model.fit(X_train, y_train)

    # MLflow 모델 레지스트리 저장
    mlflow.xgboost.log_model(final_model, "churn_model")

    # 5. 3일차 LangChain 연동용 컨텍스트 생성 (X_test 기준)
    probs = final_model.predict_proba(X_test)[:, 1]

    llm_context = pd.DataFrame({
        'customer_id': X_test.index,
        'recency_score': X_test['Recency'],
        'freq_score': X_test['Frequency'],
        'monetary_score': X_test['Monetary'],
        'churn_prob': probs
    })

    # 고위험군 상위 10명 추출 및 JSON 저장
    os.makedirs('../data/processed', exist_ok=True)
    context_sample = llm_context.sort_values(by='churn_prob', ascending=False).head(10)
    context_sample.to_json('../data/processed/llm_context.json', orient='records', indent=4)

print("2일차 수정 완료.")

데이터 로드 완료.


2026/03/31 18:44:34 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


2일차 수정 완료.
